## Basic knowledge of the LLM 


#### 1. LLM Fundamentals
####  Tokenization (BPE, WordPiece). 不是理解文本，是分词：文本--数字，不是单词级别（dis: vocabulary size很大，OOV 差）也不是 character (embedding meaning is subtle). Tokenzation is usually subwords scale. 

- token 就是 tokenizer 切出来的最小处理单位。 vocabulary（词表）就是 tokenizer 认识的所有 token 的集合。  
exp: 
unhappiness ： ["un", "happi", "ness"]
vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "I": 2,
    "love": 3,
    "play": 4,
    "##ing": 5
}
token id is [2, 3, 4, 5]  for i love playing. 

 
- The methods of the tokenizer: 
    - Byte Pair Encoding (BPE) 通过频率驱动，学出常见片段。 
    -  WordPiece WordPiece 更偏合并后对语言模型帮助大不大. Use the  score(x, y) = freq(xy) / (freq(x) * freq(y)) 
    - SentencePiece usually for the chinese character 
    - Unigram 原始文本当作字节流/字符串处理， unigram 一个词可以由若干候选子词组成，选择概率最大的切分方式。 
        - exp: 
        unhappiness split as ["un", "happi", "ness"], then count the frequency of each part find the maximum frequency split. 


 #### Embeddings (word vs contextual): Embedding it to embed the token_id to the high-dimensional vectors. The word with similar meaning will be closed in the vector space. 

    - Word embedding: the word is mapped to a **fixed vector** , which not related to the context
    methods like : word2vec, FastText 
    - Contextual embedding: usually come from the **output.last_hidden_state**, such as BERT. 
   
        - They use the nn.Embedding to initialize the (seq_len, d_model) vector, the weights can be optimized with the neural network training. 

        - The first step for the neural network , which usually has the token embedding and the position embedding : token_embedding + positional_encoding  



 ####  Positional encoding : Transformer can't tell the order as RNN which computes in the order. Transformer needs the positional embedding to encode the position information 

    - Not learnable : Absolute position embedding with sinusodial 
    - Learnable :  
        - Absolute position: nn.Embedding train the parameters. 

        - Relative position embedding. The attention will be QK + bias[i - j]  =  qk +  learned relative bias  

        - Rotational position embedding: the QK is applied with the rotation. then it only relate to the R[i-j] the relative position . 对 Q 和 K 做与位置相关的旋转, 这样 attention 分数里自动带有相对位置信息。  

 

 #### Training objective (causal LM, masked LM): 

    - Attention with causual LM : to predict the next token. It must have the casual mask and the pad mask 
        - Loss: Error on the prediction of the tokens 

    - Attention in the Masked LM : to recover the mask token, It has the musk for the musked token and the pad mask. 

        - Loss:  Error on the recover 
 

    - **How to operator the mask in the attention mechanism** 

    
    1. The Causual is to learn P(x1, x2... xt ) = p(xt| x1, x2 ... xt)
        - causal attention mask : torch.tril( torch.ones( seq_len, seq_len))
    2. The token is masked as [MASK] (token mask) 
        - only the token is masked as [MASK]. 
    3. Usually pad_token_id = 0 : 
        **with the pad_mask = ( input_ids != pad_token_id).int() return the pad_mask** 

    
    4. Loss:  
        Casual LM 
            - the model output logits with shape of (batch_size, max_length, vocab_size) 
            - label is (batch_size, squence length ) 

            - cross entropy: 
            loss_fn = nn.CrossEntropyLoss() 
            loss = loss_fn( logits.view(-1, vocabular_size), labels.review(-1)) 

        
        Masked LM 
            - prediction .
            - label is [ -100, -100, deep_id, -100] ,those -100 is neglected. 
            - cross entropy: 
            loss_fn = nn.CrossEntropyLoss(ignore_index = -100) 
            loss = loss_fn( logits.view(-1, seqence_length), true.view(-1))


 ####  Mask 
    1. How to apply the mask when operates in the attention computation 
    2. How to apply the mask in the Loss computation 

    - Attn( K, Q, V) = softmax(QKT+mask)V 
    - : that is softmax(-infinity) = 0  mask( -inf ) the weight is zero 

    - exp: (1 - mask ) * (-1e9). the mask zero means mask, with this transform, the 0 will be the -inf , which will be neglected during the attention score with the softmax 
            - code is : masked_score = score + (1 - mask) * (-1e9)
            - F.softmax( masked_score, dim = -1 )
        

        - Casual mask formula:  （1 - mask）*（-1e9） masked_fill( mask == 0, -1e9) 意思一样，都是把0 变成很大的负数
            - code is : 
                - mask = torch.tril( torch.ones(seq_len, seq_len))
                - scores = Q @ K.T 
                - scores.masked_fill( mask == 0, -1e9) # change those mask = 0 to be -1e9.  
                - attm = softmax(scores) 

        - Pad mask : the same masked_fill( mask == 0, -1e9) 


    - Loss: 
        - pad mask , token_ids = PAD will be turn into the -100, and the loss_fn = nn.CrossEntropy(ignore_index = -100). Same as the maskedLM loss computation(difference on those unmasked will be labeld as -100)

- Transformer architecture (cross-attention, self-attention, multi-head). 

        - cross-attention :  

    

    - Multi-head 



# Tokenizer operation 

In [ ]:
import torch 
import torch.nn as nn 
from transformers import AutoTokenizer

In [ ]:
# HuggingFace tokenizer  


model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name) 

text = "Hello, how are you?"
# encoded is a dict with keys 'input_ids', 'token_type_ids', 'attention_mask' 
encoded = tokenizer(text) # with the return_tensors = 'pt', the output will be tensor format 



tokens = tokenizer.tokenize( text )
print("Encoded:", encoded)
print("Tokens:", tokens)


#If use the tokenizer with return_tensors = 'pt', output will be in tensor, the shape will be (batch, seq_len )


In [ ]:
# the input ids include the special tokens such as cls, sep, etc.
# token_type_ids are used for distinguishing between different sentences in tasks like question answering, where you have a question and a context. In this case, since we only have one sentence, all token_type_ids will be 0.
# attention_mask is used to indicate which tokens should be attended to (1) and which should be ignored (0). In this case, all tokens are attended to, so the attention_mask will be all 1s.
# 0 is mask (padding), 1 is not mask,. 
input_ids = encoded['input_ids']
attention_mask = encoded['attention_mask'] 
token_type_ids = encoded['token_type_ids'] 

# Convert input_ids to the token strings 
tokens2 = tokenizer.convert_ids_to_tokens(input_ids) 
print("Tokens:", tokens2) 
print("Input IDs:", input_ids)
print("Attention Mask:", attention_mask)
print("Token Type IDs:", token_type_ids) 


In [ ]:
## if padding is needed, we can  padding in the tokenizer call.
text2 = ["Hello, how are you?", "I am fine, thank you!"]
encoded2 = tokenizer(text2, padding='max_length', max_length=10, truncation=True)  

input_ids2 = encoded2['input_ids'] 
token_type_ids2 = encoded2['token_type_ids'] 
attention_mask2 = encoded2['attention_mask']
#tokens = encoded.convert_ids_to_tokens(input_ids) 
print("Encoded:", encoded2) 
print("Input IDs:", input_ids2)
print("Attention Mask:", attention_mask2 )
print("Token Type IDs:", token_type_ids)


In [ ]:
# List of sentences: the convert to the tokens 
tokens2 = tokenizer.batch_decode(input_ids2)
print("Tokens:", tokens2)



# tokenizer.batch_decoder or convert_ids_to_tokens can be used to convert the input ids back to the tokens. 
# The batch_decode will return the string, while convert_ids_to_tokens will return the list of tokens.
 



# Embedding
- nn.Embedding (vocabular_size, d_model) , return  X input(batch, number of token) then  (batch, number of tokens, d_model).  # number of token id .
- nn.Embedding for the position, it use the (max_length, d_model )

In [ ]:
# embedding

vocab_size = 5 #[0, 1, 2, 3, 4]
embedding_dim = 3
# the embeding layer is wit the shape  vocab_size x embedding_dim
# nn.Embedding is a lookup table that maps each token index to a dense vector of fixed size
# (embedding_dim). The vocab_size is the number of unique tokens in the vocabulary, and embedding_dim is the size of the embedding vector for each token. 

embedding = nn.Embedding(vocab_size, embedding_dim) 
# a matrix of size (batch_size, vocab_size, embedding_dim) is returned when we pass the input_ids through the embedding layer.
input_ids = torch.tensor([0, 2, 1, 2, 2, 3]) 
output = embedding(input_ids) 

print("Output shape:", output.shape)
print("Output:", output )

print(embedding.weight.shape)             
print(embedding.weight.requires_grad)  
print(list(embedding.parameters())[0].shape) 

# Position embedding  

In [ ]:
# absolute position 
import math 

def pos_encoding(seq_len, d_model):    
    # pos is the index of the position in the sequence, i is the index of the dimension in the model 
    # d is the dimension of the model, and 10000 is a constant that determines the frequency of the sine and cosine functions. 
    # return the pe should be the same dimension as the embdding 
    pe = torch.zeros(seq_len, d_model)   
    
    for pos in range(seq_len): 
        for i in range(0, d_model, 2): 
            pe[pos, i] = math.sin(pos/ (10000**(i/d_model)))
            pe[pos, i+1] = math.cos(pos/ (10000**(i/d_model)))
    return pe       
    

In [ ]:
# Learned position embedding 
seq_len = 10
d_model = 4
# the nn.embedding can also work for hte position 
pos_embedding = nn.Embedding(seq_len, d_model)
# position is the index from 0 to seq_len-1, which is the position of the token in the sequence. 
positions = torch.arange(seq_len)
pos_embeded = pos_embedding(positions) 

In [ ]:
# Relative postion embedding: Attnetion only depends on the relative position of the tokens,
# not the absolute position. 
 
def 


In [ ]:
# Rotary positional embedding : with the rotation,
# it proves that the attention is only related to the relative position of the tokens, not the absolute position. 
 

def   




## LLM Casual and masked difference

In [ ]:
# Casual LLM   
from transformers import AutoTokenizer, AutoModelForCasualLM  

tokenizer = AutoTokenizer.from_pretrained(model_name) 
model = AutoModelForCasualLM.from_pretrained(model_name) 

input_text = 'I love NLP'
tokens = tokenizer(input_text, return_tensors = 'pt') 

tokens_id = tokens['input_ids']

# to train the model, input and output, input is **tokens, label is the token_ids 
output = model(**tokens, labels = tokens['input_ids']) # auto shift in the model 

loss = output.loss 
logits = output.logits 

#loss_fn = nn.CrossEntropy() 
#loss = loss_fn( output.view(-1, vocabular_size) , tokens["input_ids"])


In [ ]:
## Masked LLM 
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(model_name) 
model = AutoModelForMaskedLM.from_pretrained(model_name) 

input_text = 'I love [MASK] learning'

tokens = tokenizer(input_text, return_tensors ='pt' )

tokens_id = tokens['input_ids']
labels = tokens['input_ids'].clone() 

#get the masked token, for the loss computation 
mask_token_ids = tokenizer.mask_token_ids 
labels[ labels['input_ids'] != mask_token_ids] = -100  

outputs = model( **tokens, labels = labels  )# , ignore_index = -100  is naturally 


loss = output.loss

logits = output.logits 


## Attention 

In [ ]:
# self attention 
import torch
import torch.nn.functional as F

X = torch.randn(4, 8)  # 4 tokens, dim=8

W_Q = torch.randn(8, 8) # x feature dimension input and output 
W_K = torch.randn(8, 8)
W_V = torch.randn(8, 8)

# X W (4, 8) (8, 8) = (4, 8)
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
# score length is (4, 4)
scores = Q @ K.T / (8 ** 0.5)
# (4, 4)
weights = F.softmax(scores, dim=-1)
# (4, 4) ( 4, 8)
output = weights @ V
print(output.shape)  # [4, 8] 